# 10.08 - SSL augmentation ablation and k-NN evaluation

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** SSL augmentation ablation and k-NN evidence table.

This third self-supervised lesson emphasizes evaluation: normalize frozen embeddings, measure positive-pair alignment, and use k-nearest neighbors without training another classifier.

## Core Ideas

Two augmented views should produce similar embeddings without making every image identical in representation space. Positive-pair alignment measures one part of that behavior. A k-NN probe gives a fast, non-parametric check of whether class neighborhoods emerge. Compare augmentation settings on the same train/validation identities and report the untouched validation support.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.neighbors import KNeighborsClassifier

SEED = 10
rng = np.random.default_rng(SEED)

## Prepared Frozen Embeddings

Three class centers produce deterministic train and validation embeddings. Weak and strong view settings use the same identities; only augmentation noise changes.

In [ ]:
class_centers = np.eye(3, dtype=np.float32)
train_labels = np.repeat(np.arange(3), 12)
validation_labels = np.repeat(np.arange(3), 6)
base_train = class_centers[train_labels] + rng.normal(0.0, 0.08, size=(36, 3)).astype(np.float32)
base_validation = class_centers[validation_labels] + rng.normal(0.0, 0.08, size=(18, 3)).astype(np.float32)
weak_view_a = base_validation + rng.normal(0.0, 0.03, size=base_validation.shape).astype(np.float32)
weak_view_b = base_validation + rng.normal(0.0, 0.03, size=base_validation.shape).astype(np.float32)
strong_view_a = base_validation + rng.normal(0.0, 0.20, size=base_validation.shape).astype(np.float32)
strong_view_b = base_validation + rng.normal(0.0, 0.20, size=base_validation.shape).astype(np.float32)
print("train/validation:", base_train.shape, base_validation.shape, "validation support:", np.bincount(validation_labels).tolist())

## Exercise 10-A: Normalize embeddings

Normalize every row independently and reject zero-norm rows instead of producing NaNs.

**Return structure — `normalize_embeddings`:** A NumPy `float32` array with the same `[N,D]` shape as the input. Every row has L2 norm approximately 1.

In [ ]:
# TODO 10-A
def normalize_embeddings(embeddings):
    raise NotImplementedError("Complete Exercise 10-A")


# Smoke check: normalize the prepared training features.
normalized_train = normalize_embeddings(base_train)
print("normalized:", normalized_train.shape, np.linalg.norm(normalized_train, axis=1)[:3])

## Exercise 10-B: Measure positive-pair alignment

Cosine alignment is the mean cosine similarity between matching rows from two views. Higher is not automatically better if representations collapse.

**Return structure — `positive_pair_alignment`:** A Python `float` in `[-1,1]`. Inputs must share shape `[N,D]`.

In [ ]:
# TODO 10-B
def positive_pair_alignment(view_a, view_b):
    raise NotImplementedError("Complete Exercise 10-B")


# Smoke check: compare weak and strong view alignment.
weak_alignment = positive_pair_alignment(weak_view_a, weak_view_b)
strong_alignment = positive_pair_alignment(strong_view_a, strong_view_b)
print("weak/strong alignment:", weak_alignment, strong_alignment)

## Exercise 10-C: Run a k-NN probe

Fit only on training embeddings and labels. Validation embeddings remain untouched and appear exactly once in the confusion matrix.

**Return structure — `knn_probe`:** A dictionary with `predictions` as an integer NumPy array `[N_val]`, `macro_f1` as a Python float, `confusion_matrix` as an integer NumPy array `[C,C]`, and `validation_support` as `list[int]` of length `C`.

In [ ]:
# TODO 10-C
def knn_probe(train_embeddings, train_targets, validation_embeddings, validation_targets, k=3):
    raise NotImplementedError("Complete Exercise 10-C")


# Smoke check: evaluate the prepared frozen representation.
weak_probe = knn_probe(base_train, train_labels, weak_view_a, validation_labels, k=3)
strong_probe = knn_probe(base_train, train_labels, strong_view_a, validation_labels, k=3)
print("weak probe:", weak_probe)

## Exercise 10-D: Build an ablation table

Keep the first configuration as the baseline and expose metric changes directly.

**Return structure — `ssl_ablation_table`:** A `pandas.DataFrame` with one row per configuration and columns `configuration`, `train_size`, `validation_size`, `validation_support`, `alignment`, `macro_f1`, and `delta_macro_f1`.

In [ ]:
# TODO 10-D
def ssl_ablation_table(records):
    raise NotImplementedError("Complete Exercise 10-D")


# Smoke check and full prepared-split evidence.
ssl_evidence = ssl_ablation_table([
    {"configuration": "weak", "train_size": len(train_labels), "validation_size": len(validation_labels), "validation_support": weak_probe["validation_support"], "alignment": weak_alignment, "macro_f1": weak_probe["macro_f1"]},
    {"configuration": "strong", "train_size": len(train_labels), "validation_size": len(validation_labels), "validation_support": strong_probe["validation_support"], "alignment": strong_alignment, "macro_f1": strong_probe["macro_f1"]},
])
print(ssl_evidence.to_string(index=False))

## Test Cases

**Return structure — `run_day10_tests`:** Returns `None`; assertions and `Day 10 tests passed` communicate success.

In [ ]:
def run_day10_tests():
    assert normalized_train.shape == base_train.shape and normalized_train.dtype == np.float32
    assert np.allclose(np.linalg.norm(normalized_train, axis=1), 1.0, atol=1e-5)
    assert -1.0 <= weak_alignment <= 1.0 and -1.0 <= strong_alignment <= 1.0
    assert weak_probe["predictions"].shape == validation_labels.shape
    assert weak_probe["confusion_matrix"].shape == (3, 3)
    assert int(weak_probe["confusion_matrix"].sum()) == len(validation_labels)
    assert weak_probe["validation_support"] == [6, 6, 6]
    assert ssl_evidence.shape == (2, 7)
    assert float(ssl_evidence.iloc[0]["delta_macro_f1"]) == 0.0
    print("Day 10 tests passed")


run_day10_tests()

## Day 10 Checklist

- [ ] Normalize embeddings before cosine or k-NN comparisons.
- [ ] Measure paired-view alignment without confusing it with class quality.
- [ ] Fit k-NN on training identities only.
- [ ] Report full validation support, Macro-F1, and baseline delta.
- [ ] Run the test cases.